### Reranking Hybrid Search Statergies

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.

👉 It ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [21]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

In [22]:
# Create sample documents instead of loading from file
sample_text = """
LangChain is a framework for building applications with large language models.
It provides tools and abstractions for working with LLMs, document loading, and more.

LangChain supports memory functionality to maintain context across conversations.
You can use different memory types like ConversationBufferMemory, ConversationSummaryMemory.

Tools in LangChain allow LLMs to interact with external systems and APIs.
Examples include search tools, calculators, and custom functions.

To build applications with memory and tools, you can create agents that use both.
Agents can remember previous interactions and call tools when needed.

Vector stores in LangChain help with document retrieval and similarity search.
Popular vector stores include FAISS, Chroma, and Pinecone.

Text splitters help break down large documents into manageable chunks.
This is essential for embedding and retrieval in RAG applications.
"""

# Create documents from the sample text
from langchain_core.documents import Document
raw_docs = [Document(page_content=sample_text, metadata={"source": "sample"})]

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
print(f"Created {len(docs)} document chunks")
docs

Created 2 document chunks


[Document(metadata={'source': 'sample'}, page_content='LangChain is a framework for building applications with large language models.\nIt provides tools and abstractions for working with LLMs, document loading, and more.\n\nLangChain supports memory functionality to maintain context across conversations.\nYou can use different memory types like ConversationBufferMemory, ConversationSummaryMemory.\n\nTools in LangChain allow LLMs to interact with external systems and APIs.\nExamples include search tools, calculators, and custom functions.'),
 Document(metadata={'source': 'sample'}, page_content='To build applications with memory and tools, you can create agents that use both.\nAgents can remember previous interactions and call tools when needed.\n\nVector stores in LangChain help with document retrieval and similarity search.\nPopular vector stores include FAISS, Chroma, and Pinecone.\n\nText splitters help break down large documents into manageable chunks.\nThis is essential for embedd

In [23]:
## user query
query="How can i use langchain to build an application with memory and tools?"

In [24]:
### FAISS and Huggingface model Embeddings

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7772.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
## OpenAI Embedding
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings

embeddings=OpenAIEmbeddings()
vectorstore_openai=FAISS.from_documents(docs,embeddings)
retriever_openai=vectorstore_openai.as_retriever(search_kwargs={"k":8})




In [28]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020C02349450>, search_kwargs={'k': 8})

In [29]:
retriever_openai

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020C0240C180>, search_kwargs={'k': 8})

In [30]:
## prompt and use the llm
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.2)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020C02349810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020C0234A490>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [32]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [33]:
retrieved_docs=retriever.invoke(query)
retrieved_docs

[Document(id='270c6623-06ce-431f-9cf3-9c8bbebd9466', metadata={'source': 'sample'}, page_content='LangChain is a framework for building applications with large language models.\nIt provides tools and abstractions for working with LLMs, document loading, and more.\n\nLangChain supports memory functionality to maintain context across conversations.\nYou can use different memory types like ConversationBufferMemory, ConversationSummaryMemory.\n\nTools in LangChain allow LLMs to interact with external systems and APIs.\nExamples include search tools, calculators, and custom functions.'),
 Document(id='d19aa063-0294-49bd-9206-8e45c29e98d7', metadata={'source': 'sample'}, page_content='To build applications with memory and tools, you can create agents that use both.\nAgents can remember previous interactions and call tools when needed.\n\nVector stores in LangChain help with document retrieval and similarity search.\nPopular vector stores include FAISS, Chroma, and Pinecone.\n\nText splitters

In [34]:
chain=prompt| llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020C02349810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020C0234

In [35]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [36]:
doc_lines

['1. LangChain is a framework for building applications with large language models.\nIt provides tools and abstractions for working with LLMs, document loading, and more.\n\nLangChain supports memory functionality to maintain context across conversations.\nYou can use different memory types like ConversationBufferMemory, ConversationSummaryMemory.\n\nTools in LangChain allow LLMs to interact with external systems and APIs.\nExamples include search tools, calculators, and custom functions.',
 '2. To build applications with memory and tools, you can create agents that use both.\nAgents can remember previous interactions and call tools when needed.\n\nVector stores in LangChain help with document retrieval and similarity search.\nPopular vector stores include FAISS, Chroma, and Pinecone.\n\nText splitters help break down large documents into manageable chunks.\nThis is essential for embedding and retrieval in RAG applications.']

In [37]:
formatted_docs

'1. LangChain is a framework for building applications with large language models.\nIt provides tools and abstractions for working with LLMs, document loading, and more.\n\nLangChain supports memory functionality to maintain context across conversations.\nYou can use different memory types like ConversationBufferMemory, ConversationSummaryMemory.\n\nTools in LangChain allow LLMs to interact with external systems and APIs.\nExamples include search tools, calculators, and custom functions.\n2. To build applications with memory and tools, you can create agents that use both.\nAgents can remember previous interactions and call tools when needed.\n\nVector stores in LangChain help with document retrieval and similarity search.\nPopular vector stores include FAISS, Chroma, and Pinecone.\n\nText splitters help break down large documents into manageable chunks.\nThis is essential for embedding and retrieval in RAG applications.'

In [38]:
response=chain.invoke({"question":query,"documents":formatted_docs})
response

"Based on the user's question, I would rank the documents as follows:\n\nThe most relevant document is the one that directly mentions using LangChain to build an application with memory and tools. \n\nThe second most relevant document is the one that explains how to build applications with memory and tools using LangChain.\n\nThe third most relevant document is the one that explains the tools available in LangChain, which can be used to interact with external systems and APIs.\n\nThe least relevant document is the one that talks about vector stores, text splitters, and their applications, which are not directly related to building an application with memory and tools.\n\nSo, the ranked list of document indices is: 1, 2, 0"

In [39]:
# Step 5: Parse and rerank
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
indices

[1, -1]

In [40]:
retrieved_docs

[Document(id='270c6623-06ce-431f-9cf3-9c8bbebd9466', metadata={'source': 'sample'}, page_content='LangChain is a framework for building applications with large language models.\nIt provides tools and abstractions for working with LLMs, document loading, and more.\n\nLangChain supports memory functionality to maintain context across conversations.\nYou can use different memory types like ConversationBufferMemory, ConversationSummaryMemory.\n\nTools in LangChain allow LLMs to interact with external systems and APIs.\nExamples include search tools, calculators, and custom functions.'),
 Document(id='d19aa063-0294-49bd-9206-8e45c29e98d7', metadata={'source': 'sample'}, page_content='To build applications with memory and tools, you can create agents that use both.\nAgents can remember previous interactions and call tools when needed.\n\nVector stores in LangChain help with document retrieval and similarity search.\nPopular vector stores include FAISS, Chroma, and Pinecone.\n\nText splitters

In [41]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='d19aa063-0294-49bd-9206-8e45c29e98d7', metadata={'source': 'sample'}, page_content='To build applications with memory and tools, you can create agents that use both.\nAgents can remember previous interactions and call tools when needed.\n\nVector stores in LangChain help with document retrieval and similarity search.\nPopular vector stores include FAISS, Chroma, and Pinecone.\n\nText splitters help break down large documents into manageable chunks.\nThis is essential for embedding and retrieval in RAG applications.')]

In [42]:
# Step 6: Show results
print("\n📊 Final Reranked Results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


📊 Final Reranked Results:

Rank 1:
To build applications with memory and tools, you can create agents that use both.
Agents can remember previous interactions and call tools when needed.

Vector stores in LangChain help with document retrieval and similarity search.
Popular vector stores include FAISS, Chroma, and Pinecone.

Text splitters help break down large documents into manageable chunks.
This is essential for embedding and retrieval in RAG applications.
